# DKD Risk Prediction in Type 2 Diabetes
## 03B. Model Development

This notebook trains the corrected reviewer-response models using the finalized modeling datasets and split assignments produced by Notebook 03A.

The corrected landmark cohorts are the authoritative analytic populations. All eligible corrected participants are included.

To maximize comparability with the submitted analysis while preserving an exact corrected 60%/20%/20% stratified design:

- shared participants initially retain the reconstructed manuscript partition;
- reference-only participants are removed;
- corrected-only participants are assigned deterministically with seed 42;
- only the minimum number of shared participants needed to restore the exact corrected-cohort split and case/control targets are rebalanced;
- corrected outcome labels are used throughout.

The remaining model-development framework is unchanged: progressive M1-M5 information layers, the original XGBoost search space, 5-fold stratified cross-validation within training, and isotonic calibration within the calibration partition.


# 1. Setup


In [ ]:
import json
import pickle
import warnings

import gcsfs
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
)
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings("ignore")

migration_folder = "rw-migration-aou-rw-639fade3"
bucket = migration_folder

COHORT_DIR = f"{bucket}/aou_dkd/dkd_cohort_20260821"
MODEL_DIR = f"{bucket}/aou_dkd/dkd_models_20260822"
RESULT_DIR = f"{bucket}/aou_dkd/dkd_model_results_20260822"

fs = gcsfs.GCSFileSystem()

ID_COL = "person_id"
Y_COL = "dkd_binary"

FINAL_DATE = "20260822"
SPLIT_SEED = 42
N_XGB_TRIALS = 20

XGB_BOUNDS = {
    "gamma": (1e-8, 1.0),
    "max_depth": (3, 6),
    "min_child_weight": (1, 6),
    "subsample": (0.6, 1.0),
    "colsample_bytree": (0.6, 1.0),
    "learning_rate": (0.01, 0.15),
    "n_estimators": (150, 300),
    "reg_alpha": (1e-8, 0.5),
    "reg_lambda": (1e-8, 0.5),
}

SCALE_POS_WEIGHT_MIN = 1
SCALE_POS_WEIGHT_MAX = 2

MODEL_ORDER = {
    "30d": ["M1", "M2", "M3", "M4a", "M5"],
    "6m": ["M1", "M2", "M3", "M4a", "M5"],
    "1y": ["M1", "M2", "M3", "M4", "M5"],
    "2y": ["M1", "M2", "M3", "M4", "M5"],
}

FORBIDDEN_MODEL_FEATURES = {
    "t2d_first_evidence_date",
    "cutoff_date",
    "kidney_first_evidence_date",
    "ehr_end_date",
    "landmark_name",
    "landmark_days",
    "event_or_censor_date",
    "last_followup_date",
    "first_activity_date",
    "days_kidney_after_t2d",
    "days_kidney_minus_t2d",
    "incident_case_candidate",
    "eligible_control",
    "early_kidney_event",
    "landmark_case",
    "landmark_control",
    "observable_at_landmark",
    "incident_dkd_on_or_before_landmark",
}


## I/O helpers


In [ ]:
def read_gcs_pickle(path):
    with fs.open(path, "rb") as f:
        return pd.read_pickle(f)


def read_gcs_csv(path):
    with fs.open(path, "r") as f:
        return pd.read_csv(f)


def save_gcs_pickle(obj, path):
    with fs.open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"Saved pickle: {path}")


def save_gcs_csv(df, path):
    with fs.open(path, "w") as f:
        df.to_csv(f, index=False)
    print(f"Saved CSV: {path}")


def save_gcs_json(obj, path):
    with fs.open(path, "w") as f:
        json.dump(obj, f, indent=2)
    print(f"Saved JSON: {path}")


# 2. Load Finalized Inputs from Notebook 03A


In [ ]:
FINAL_MODEL_PATHS = {
    landmark: (
        f"{COHORT_DIR}/"
        f"DKD_Risk_FinalModel_Data_Landmark_"
        f"{landmark}_{FINAL_DATE}.pkl"
    )
    for landmark in [
        "30d",
        "6m",
        "1y",
        "2y",
    ]
}

FEATURE_MANIFEST_PATH = (
    f"{RESULT_DIR}/"
    f"final_model_feature_manifest_{FINAL_DATE}.csv"
)

SPLIT_MANIFEST_PATH = (
    f"{RESULT_DIR}/"
    f"final_model_split_manifest_{FINAL_DATE}.csv"
)

for path in (
    list(
        FINAL_MODEL_PATHS.values()
    )
    + [
        FEATURE_MANIFEST_PATH,
        SPLIT_MANIFEST_PATH,
    ]
):
    if not fs.exists(path):
        raise FileNotFoundError(path)

landmark_data = {
    landmark: read_gcs_pickle(path)
    for landmark, path in FINAL_MODEL_PATHS.items()
}

feature_manifest = read_gcs_csv(
    FEATURE_MANIFEST_PATH
)

split_manifest = read_gcs_csv(
    SPLIT_MANIFEST_PATH
)

print("Final reviewer-response landmark datasets")
for landmark, df in landmark_data.items():
    print(
        f"  {landmark:>3}: "
        f"N={len(df):,} | "
        f"cases={int(df[Y_COL].sum()):,} | "
        f"columns={df.shape[1]:,}"
    )


# 3. Rebuild and Validate the Model-Specific Matrices


In [ ]:
model_features = {}
model_data = {}

for landmark in MODEL_ORDER:
    model_features[landmark] = {}
    model_data[landmark] = {}

    for model_id in MODEL_ORDER[landmark]:
        manifest = (
            feature_manifest.loc[
                feature_manifest[
                    "landmark"
                ].eq(
                    landmark
                )
                & feature_manifest[
                    "model_id"
                ].eq(
                    model_id
                )
            ]
            .sort_values(
                "feature_order"
            )
        )

        features = manifest[
            "feature"
        ].tolist()

        if not features:
            raise ValueError(
                f"{landmark}/{model_id}: no feature manifest."
            )

        forbidden = sorted(
            set(features)
            & FORBIDDEN_MODEL_FEATURES
        )

        if forbidden:
            raise ValueError(
                f"{landmark}/{model_id}: forbidden cohort/outcome "
                f"metadata in feature manifest: {forbidden}"
            )

        missing = [
            feature
            for feature in features
            if feature not in landmark_data[
                landmark
            ].columns
        ]

        if missing:
            raise ValueError(
                f"{landmark}/{model_id}: final data missing "
                f"manifest predictors: {missing}"
            )

        model_df = landmark_data[
            landmark
        ].loc[
            :,
            [
                ID_COL,
                Y_COL,
            ]
            + features,
        ].copy()

        model_features[
            landmark
        ][
            model_id
        ] = features

        model_data[
            landmark
        ][
            model_id
        ] = model_df

        print(
            f"{landmark:>3} | {model_id:>3} | "
            f"N={len(model_df):,} | "
            f"predictors={len(features):,}"
        )


# Progressive nesting check.
for landmark, order in MODEL_ORDER.items():
    previous = set()

    for model_id in order:
        current = set(
            model_features[
                landmark
            ][
                model_id
            ]
        )

        if not previous.issubset(
            current
        ):
            raise ValueError(
                f"{landmark}: model layers are not nested at {model_id}."
            )

        previous = current

print("\nFeature manifest and progressive model structure validated.")


# Explicit landmark-specific layer-name contract.
EXPECTED_MODEL_IDS = {
    "30d": {"M1", "M2", "M3", "M4a", "M5"},
    "6m": {"M1", "M2", "M3", "M4a", "M5"},
    "1y": {"M1", "M2", "M3", "M4", "M5"},
    "2y": {"M1", "M2", "M3", "M4", "M5"},
}

for landmark, expected_ids in EXPECTED_MODEL_IDS.items():
    actual_ids = set(
        model_features[
            landmark
        ].keys()
    )

    if actual_ids != expected_ids:
        raise ValueError(
            f"{landmark}: unexpected model IDs {sorted(actual_ids)}."
        )

print(
    "M4a at 30d/6m and M4 at 1y/2y: PASSED"
)


# 4. Load and Validate the Preserved Split Assignments


In [ ]:
split_lookup = {}
cohort_order = {}

for landmark in MODEL_ORDER:
    x = split_manifest.loc[
        split_manifest[
            "landmark"
        ].eq(
            landmark
        )
    ].copy()

    if x[ID_COL].duplicated().any():
        raise ValueError(
            f"{landmark}: duplicate person_id in split manifest."
        )

    split_lookup[landmark] = {
        int(pid): split
        for pid, split in zip(
            x[ID_COL],
            x["split"],
        )
    }

    cohort_order[landmark] = {
        int(pid): int(order)
        for pid, order in zip(
            x[ID_COL],
            x["cohort_order"],
        )
    }

    data_ids = set(
        landmark_data[
            landmark
        ][
            ID_COL
        ].astype(int)
    )

    manifest_ids = set(
        split_lookup[
            landmark
        ].keys()
    )

    if data_ids != manifest_ids:
        raise ValueError(
            f"{landmark}: dataset/split participant mismatch."
        )

    print(
        f"{landmark:>3}: "
        f"N={len(data_ids):,} | split manifest validated"
    )


# 30d and 6m must be identical by design.
split_30d = (
    split_manifest.loc[
        split_manifest[
            "landmark"
        ].eq(
            "30d"
        ),
        [
            ID_COL,
            Y_COL,
            "split",
        ],
    ]
    .sort_values(
        ID_COL
    )
    .reset_index(
        drop=True
    )
)

split_6m = (
    split_manifest.loc[
        split_manifest[
            "landmark"
        ].eq(
            "6m"
        ),
        [
            ID_COL,
            Y_COL,
            "split",
        ],
    ]
    .sort_values(
        ID_COL
    )
    .reset_index(
        drop=True
    )
)

if not split_30d.equals(
    split_6m
):
    raise ValueError(
        "30d and 6m participants/outcomes/splits are not identical."
    )

print(
    "30d/6m participant, outcome, and split identity: PASSED"
)


## Split composition


In [ ]:
for landmark in MODEL_ORDER:
    x = split_manifest.loc[
        split_manifest[
            "landmark"
        ].eq(
            landmark
        )
    ].copy()

    tab = (
        x.groupby(
            "split"
        )[Y_COL]
        .agg(
            N="size",
            cases="sum",
        )
        .reindex(
            [
                "train",
                "cal",
                "test",
            ]
        )
    )

    tab["controls"] = (
        tab["N"]
        - tab["cases"]
    )

    print(f"\n{landmark}")
    print(tab)


# 5. Training Helpers


In [ ]:
def get_partition(
    df,
    landmark,
    split_name,
):
    ids = {
        pid
        for pid, split in split_lookup[
            landmark
        ].items()
        if split == split_name
    }

    x = df.loc[
        df[
            ID_COL
        ].astype(int).isin(
            ids
        )
    ].copy()

    x["_cohort_order"] = (
        x[
            ID_COL
        ]
        .astype(int)
        .map(
            cohort_order[
                landmark
            ]
        )
    )

    return (
        x.sort_values(
            "_cohort_order"
        )
        .drop(
            columns="_cohort_order"
        )
        .reset_index(drop=True)
    )


def prepare_xy(df):
    X = (
        df.drop(
            columns=[
                ID_COL,
                Y_COL,
            ],
            errors="ignore",
        )
        .astype(np.float32)
    )

    y = (
        df[
            Y_COL
        ]
        .astype(int)
        .to_numpy()
    )

    return X, y


# 6. XGBoost Hyperparameter Optimization, Fitting, and Calibration


In [ ]:
def train_calibrate_xgb(
    df,
    *,
    landmark,
    model_id,
    n_trials=N_XGB_TRIALS,
):
    train_df = get_partition(
        df,
        landmark,
        "train",
    )

    cal_df = get_partition(
        df,
        landmark,
        "cal",
    )

    test_df = get_partition(
        df,
        landmark,
        "test",
    )

    X_train, y_train = prepare_xy(
        train_df
    )

    X_cal, y_cal = prepare_xy(
        cal_df
    )

    X_test, y_test = prepare_xy(
        test_df
    )

    expected_features = model_features[
        landmark
    ][
        model_id
    ]

    if list(
        X_train.columns
    ) != expected_features:
        raise ValueError(
            f"{landmark}/{model_id}: feature order changed."
        )

    for split_name, y in [
        ("train", y_train),
        ("cal", y_cal),
        ("test", y_test),
    ]:
        if np.unique(y).size != 2:
            raise ValueError(
                f"{landmark}/{model_id}: "
                f"{split_name} contains only one outcome class."
            )

    print(
        f"\nTraining {landmark} {model_id} "
        f"| predictors={len(expected_features):,}"
    )

    print(
        f"  train: N={len(y_train):,}, "
        f"cases={int(y_train.sum()):,}"
    )

    print(
        f"  cal:   N={len(y_cal):,}, "
        f"cases={int(y_cal.sum()):,}"
    )

    print(
        f"  test:  N={len(y_test):,}, "
        f"cases={int(y_test.sum()):,}"
    )

    def objective(trial):
        params = {
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "tree_method": "hist",
            "gamma": trial.suggest_float(
                "gamma",
                *XGB_BOUNDS[
                    "gamma"
                ],
                log=True,
            ),
            "max_depth": trial.suggest_int(
                "max_depth",
                *XGB_BOUNDS[
                    "max_depth"
                ],
            ),
            "min_child_weight": trial.suggest_int(
                "min_child_weight",
                *XGB_BOUNDS[
                    "min_child_weight"
                ],
            ),
            "subsample": trial.suggest_float(
                "subsample",
                *XGB_BOUNDS[
                    "subsample"
                ],
            ),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree",
                *XGB_BOUNDS[
                    "colsample_bytree"
                ],
            ),
            "learning_rate": trial.suggest_float(
                "learning_rate",
                *XGB_BOUNDS[
                    "learning_rate"
                ],
            ),
            "n_estimators": trial.suggest_int(
                "n_estimators",
                *XGB_BOUNDS[
                    "n_estimators"
                ],
            ),
            "reg_alpha": trial.suggest_float(
                "reg_alpha",
                *XGB_BOUNDS[
                    "reg_alpha"
                ],
                log=True,
            ),
            "reg_lambda": trial.suggest_float(
                "reg_lambda",
                *XGB_BOUNDS[
                    "reg_lambda"
                ],
                log=True,
            ),
            "scale_pos_weight": trial.suggest_float(
                "scale_pos_weight",
                SCALE_POS_WEIGHT_MIN,
                SCALE_POS_WEIGHT_MAX,
            ),
            "n_jobs": 16,
            "random_state": SPLIT_SEED,
        }

        model = xgb.XGBClassifier(
            **params
        )

        cv = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=SPLIT_SEED,
        )

        scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring="roc_auc",
            n_jobs=5,
        )

        return float(
            scores.mean()
        )

    sampler = optuna.samplers.TPESampler(
        seed=SPLIT_SEED
    )

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
    )

    study.optimize(
        objective,
        n_trials=n_trials,
    )

    selected_params = dict(
        study.best_params
    )

    fit_params = {
        **selected_params,
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "n_jobs": 64,
        "random_state": SPLIT_SEED,
    }

    model = xgb.XGBClassifier(
        **fit_params
    )

    model.fit(
        X_train,
        y_train,
    )

    fitted_features = list(
        model.get_booster().feature_names
    )

    if fitted_features != expected_features:
        raise ValueError(
            f"{landmark}/{model_id}: fitted feature order changed."
        )

    calibrator = CalibratedClassifierCV(
        model,
        method="isotonic",
        cv="prefit",
    )

    calibrator.fit(
        X_cal,
        y_cal,
    )

    stem = (
        f"dkd_{landmark}_{model_id}"
    )

    raw_model_path = (
        f"{MODEL_DIR}/"
        f"{stem}_xgb_raw_{FINAL_DATE}.pkl"
    )

    calibrator_path = (
        f"{MODEL_DIR}/"
        f"{stem}_isotonic_calibrator_{FINAL_DATE}.pkl"
    )

    save_gcs_pickle(
        model,
        raw_model_path,
    )

    save_gcs_pickle(
        calibrator,
        calibrator_path,
    )

    prediction_rows = []

    for split_name, split_df in [
        ("cal", cal_df),
        ("test", test_df),
    ]:
        X_split, y_split = prepare_xy(
            split_df
        )

        prediction_rows.append(
            pd.DataFrame(
                {
                    ID_COL: split_df[
                        ID_COL
                    ].astype(int).to_numpy(),
                    "split": split_name,
                    "y_true": y_split,
                    "p_raw": model.predict_proba(
                        X_split
                    )[:, 1],
                    "p_calibrated": calibrator.predict_proba(
                        X_split
                    )[:, 1],
                }
            )
        )

    predictions = pd.concat(
        prediction_rows,
        ignore_index=True,
    )

    prediction_path = (
        f"{RESULT_DIR}/"
        f"{stem}_predictions_{FINAL_DATE}.csv"
    )

    save_gcs_csv(
        predictions,
        prediction_path,
    )

    return {
        "model": model,
        "calibrator": calibrator,
        "selected_params": selected_params,
        "best_cv_auroc": float(
            study.best_value
        ),
        "n_features": len(
            expected_features
        ),
        "paths": {
            "raw_model": raw_model_path,
            "calibrator": calibrator_path,
            "predictions": prediction_path,
        },
    }


# 7. Fit All Models


In [ ]:
xgb_results = {}

for landmark in [
    "30d",
    "6m",
    "1y",
    "2y",
]:
    xgb_results[
        landmark
    ] = {}

    for model_id in MODEL_ORDER[
        landmark
    ]:
        xgb_results[
            landmark
        ][
            model_id
        ] = train_calibrate_xgb(
            model_data[
                landmark
            ][
                model_id
            ],
            landmark=landmark,
            model_id=model_id,
        )


# 8. Save Selected Hyperparameters


In [ ]:
best_param_rows = []

for landmark, models in xgb_results.items():
    for model_id, result in models.items():
        best_param_rows.append(
            {
                "landmark": landmark,
                "model_id": model_id,
                "n_features": result[
                    "n_features"
                ],
                "best_cv_auroc": result[
                    "best_cv_auroc"
                ],
                **result[
                    "selected_params"
                ],
            }
        )

best_params_df = (
    pd.DataFrame(
        best_param_rows
    )
    .sort_values(
        [
            "landmark",
            "model_id",
        ]
    )
    .reset_index(drop=True)
)

BEST_PARAMS_PATH = (
    f"{RESULT_DIR}/"
    f"xgb_best_hyperparameters_{FINAL_DATE}.csv"
)

save_gcs_csv(
    best_params_df,
    BEST_PARAMS_PATH,
)

print("\n" + "=" * 90)
print("SELECTED XGBOOST HYPERPARAMETERS")
print("=" * 90)

for _, row in best_params_df.iterrows():
    print(
        f"\n{row['landmark']} | "
        f"{row['model_id']} | "
        f"features={int(row['n_features'])} | "
        f"best CV AUROC={row['best_cv_auroc']:.5f}"
    )

display(best_params_df)


# 9. Reproducibility Summary


In [ ]:

training_summary = {
    "date": FINAL_DATE,
    "population_contract": (
        "all participants eligible under corrected landmark risk-set definitions"
    ),
    "feature_contract": (
        "archived model-specific predictor sets intersected with "
        "corrected-data feature curation"
    ),
    "corrected_only_predictors_added": False,
    "artificial_missing_levels_created": False,
    "split_contract": (
        "preserve reconstructed manuscript split where possible; "
        "add corrected-only participants with seed 42; "
        "minimally rebalance shared participants to exact corrected "
        "60/20/20 stratified targets"
    ),
    "split_seed": SPLIT_SEED,
    "xgb_trials": N_XGB_TRIALS,
    "optuna_sampler": "TPE, seed 42",
    "xgboost_random_state": SPLIT_SEED,
    "cross_validation": "5-fold stratified, seed 42",
    "calibration_method": "isotonic",
}

SUMMARY_PATH = (
    f"{RESULT_DIR}/"
    f"model_training_summary_{FINAL_DATE}.json"
)

save_gcs_json(
    training_summary,
    SUMMARY_PATH,
)

print(
    json.dumps(
        training_summary,
        indent=2,
    )
)
